In [ ]:
# !pip install yfinance
# !pip install numpy
# !pip install pandas
# !pip install scipy
# !pip install matplotlib
# !pip install tqdm

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from scipy import stats
from itertools import product as iproduct
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

print('✅ All imports successful')

---
## ⚙️ CONFIGURATION — Change these before running
---

In [ ]:
# ──────────────────────────────────────────────
#  ASSET & DATA
# ──────────────────────────────────────────────
TICKER     = 'EURUSD=X'     # Any yfinance ticker
START_DATE = '2018-01-01'
INTERVAL   = '1d'           # '1d' | '1h' (hourly needs short range)

# ──────────────────────────────────────────────
#  FTMO RULES
# ──────────────────────────────────────────────
ACCOUNT_SIZE      = 100_000
PROFIT_TARGET_PCT = 0.10    # 10%
MAX_DAILY_DD_PCT  = 0.05    # 5%
MAX_TOTAL_DD_PCT  = 0.10    # 10%
MAX_TRADING_DAYS  = 30

# ──────────────────────────────────────────────
#  BACKTEST
# ──────────────────────────────────────────────
POSITION_SIZE     = 0.02    # 2% risk per trade
SPREAD_PIPS       = 1.0
PIP_SIZE          = 0.0001
TRAIN_PCT         = 0.70    # 70/30 train/test
MIN_TRADES        = 10      # Discard combos with fewer trades

# ──────────────────────────────────────────────
#  ACTIVE STRATEGY  (choose one)
# ──────────────────────────────────────────────
ACTIVE_STRATEGY = 'EMA Cross'   # 'EMA Cross' | 'RSI Mean Revert' | 'BB Breakout' | 'MACD Trend' | 'Donchian'

print(f'✅ Config: {TICKER} | Strategy: {ACTIVE_STRATEGY} | Split: {int(TRAIN_PCT*100)}/{int((1-TRAIN_PCT)*100)}')

---
## 📥 DATA DOWNLOAD
---

In [ ]:
raw = yf.download(TICKER, start=START_DATE, interval=INTERVAL, auto_adjust=True, progress=False)

if raw.empty:
    raise ValueError(f'❌ No data downloaded for {TICKER}')

# Flatten MultiIndex columns if present
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

raw.index = pd.to_datetime(raw.index)
if raw.index.tz is not None:
    raw.index = raw.index.tz_localize(None)

raw = raw[['Open','High','Low','Close','Volume']].dropna()

print(f'Successfully downloaded {len(raw):,} records for {TICKER} from {START_DATE}')
print(f'Data range: {raw.index.min().date()} to {raw.index.max().date()}')
print(f'\nFirst 5 rows:')
display(raw.head())
raw

---
## ✂️ TRAIN / TEST SPLIT  (70 / 30)
---

In [ ]:
split_idx  = int(len(raw) * TRAIN_PCT)
train_data = raw.iloc[:split_idx].copy()
test_data  = raw.iloc[split_idx:].copy()

train_close = train_data['Close']
test_close  = test_data['Close']

print(f'Data ready:')
print(f'  Train  ({int(TRAIN_PCT*100)}%): {train_data.index[0].date()} → {train_data.index[-1].date()}  ({len(train_data):,} bars)')
print(f'  Test   ({int((1-TRAIN_PCT)*100)}%): {test_data.index[0].date()}  → {test_data.index[-1].date()}  ({len(test_data):,} bars)')

---
## 🧮 INDICATOR LIBRARY  (pure numpy — no external TA libraries)
---

In [ ]:
def _ema(close, period):
    return close.ewm(span=period, adjust=False).mean()

def _atr(high, low, close, period=14):
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1/period, adjust=False).mean()

def _rsi(close, period=14):
    delta = close.diff()
    gain  = delta.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    return 100 - 100 / (1 + gain / loss.replace(0, 1e-9))

def _bbands(close, period=20, std_mult=2.0):
    ma = close.rolling(period).mean()
    sd = close.rolling(period).std()
    return ma + std_mult * sd, ma - std_mult * sd

def _macd(close, fast=12, slow=26, signal=9):
    macd = close.ewm(span=fast, adjust=False).mean() - close.ewm(span=slow, adjust=False).mean()
    sig  = macd.ewm(span=signal, adjust=False).mean()
    return macd, sig

def _adx(high, low, close, period=14):
    up, down = high.diff(), -low.diff()
    pdm = up.where((up > down) & (up > 0), 0.0)
    ndm = down.where((down > up) & (down > 0), 0.0)
    atr_v = _atr(high, low, close, period)
    s = lambda x: x.ewm(alpha=1/period, adjust=False).mean()
    pdi = 100 * s(pdm) / atr_v.replace(0, 1e-9)
    ndi = 100 * s(ndm) / atr_v.replace(0, 1e-9)
    dx  = 100 * (pdi - ndi).abs() / (pdi + ndi).replace(0, 1e-9)
    return dx.ewm(alpha=1/period, adjust=False).mean()

print('✅ Indicator library ready (ATR, RSI, BB, MACD, ADX — zero dependencies)')

---
## 📐 STRATEGY SIGNAL GENERATORS
---

In [ ]:
def signal_ema_cross(data, fast, slow, atr_period=14, atr_mult=1.0):
    """Triple EMA crossover with ATR volatility filter."""
    e1  = _ema(data['Close'], fast)
    e2  = _ema(data['Close'], slow)
    atr = _atr(data['High'], data['Low'], data['Close'], atr_period)
    vol_ok = atr > atr.rolling(50).mean() * atr_mult
    raw = np.where(e1 > e2, 1, -1)
    return pd.Series(raw * vol_ok.astype(int).values, index=data.index)

def signal_rsi_mean_revert(data, rsi_period=14, oversold=30, overbought=70, ema_trend=200):
    """RSI mean reversion with EMA trend filter."""
    rsi   = _rsi(data['Close'], rsi_period)
    trend = _ema(data['Close'], ema_trend)
    sig   = pd.Series(0, index=data.index)
    sig[(rsi < oversold)   & (data['Close'] > trend)] =  1
    sig[(rsi > overbought) & (data['Close'] < trend)] = -1
    return sig

def signal_bb_breakout(data, bb_period=20, bb_std=2.0, rsi_period=14, rsi_thresh=50):
    """Bollinger Band breakout with RSI confirmation."""
    upper, lower = _bbands(data['Close'], bb_period, bb_std)
    rsi = _rsi(data['Close'], rsi_period)
    sig = pd.Series(0, index=data.index)
    sig[(data['Close'] > upper) & (rsi > rsi_thresh)] =  1
    sig[(data['Close'] < lower) & (rsi < rsi_thresh)] = -1
    return sig

def signal_macd_trend(data, fast=12, slow=26, signal=9, adx_period=14, adx_thresh=25):
    """MACD crossover with ADX trend filter."""
    macd_l, sig_l = _macd(data['Close'], fast, slow, signal)
    adx = _adx(data['High'], data['Low'], data['Close'], adx_period)
    strong = adx > adx_thresh
    s = pd.Series(0, index=data.index)
    s[(macd_l > sig_l) & strong] =  1
    s[(macd_l < sig_l) & strong] = -1
    return s

def signal_donchian(data, period=20, exit_period=10):
    """Donchian channel breakout."""
    upper = data['High'].rolling(period).max()
    lower = data['Low'].rolling(period).min()
    ex_up = data['High'].rolling(exit_period).max()
    ex_lo = data['Low'].rolling(exit_period).min()
    sig = pd.Series(0, index=data.index)
    pos = 0
    for i in range(period, len(data)):
        p = data['Close'].iloc[i]
        if pos ==  0:
            if p >= upper.iloc[i-1]:  pos =  1
            elif p <= lower.iloc[i-1]: pos = -1
        elif pos ==  1 and p <= ex_lo.iloc[i-1]: pos = 0
        elif pos == -1 and p >= ex_up.iloc[i-1]: pos = 0
        sig.iloc[i] = pos
    return sig

STRATEGY_SIGNALS = {
    'EMA Cross'       : signal_ema_cross,
    'RSI Mean Revert' : signal_rsi_mean_revert,
    'BB Breakout'     : signal_bb_breakout,
    'MACD Trend'      : signal_macd_trend,
    'Donchian'        : signal_donchian,
}

print(f'✅ {len(STRATEGY_SIGNALS)} strategies ready: {list(STRATEGY_SIGNALS.keys())}')

---
## 🏗️ FTMO BACKTESTER
Enforces 5% daily DD and 10% total DD rules mid-simulation.

---

In [ ]:
def run_ftmo_backtest(data, signals, position_size=POSITION_SIZE,
                      spread_pips=SPREAD_PIPS, pip_size=PIP_SIZE,
                      account=ACCOUNT_SIZE,
                      max_daily_dd=MAX_DAILY_DD_PCT,
                      max_total_dd=MAX_TOTAL_DD_PCT):
    """
    Vectorized FTMO backtest with mid-simulation DD rule enforcement.
    Returns equity Series and trades list.
    """
    close   = data['Close'].values
    sig     = signals.values
    n       = len(close)
    spread_cost = spread_pips * pip_size

    equity      = np.zeros(n)
    equity[0]   = account
    peak_equity = account
    day_start_equity = account

    trades      = []
    position    = 0        # 1=long, -1=short, 0=flat
    entry_price = 0.0
    entry_idx   = 0
    halted      = False

    prev_date = data.index[0].date()

    for i in range(1, n):
        cur_date = data.index[i].date()

        # New day reset
        if cur_date != prev_date:
            day_start_equity = equity[i-1]
            prev_date = cur_date

        equity[i] = equity[i-1]

        if halted:
            continue

        # Mark-to-market open position
        if position != 0:
            price_change = close[i] - close[i-1]
            pnl = position * price_change / pip_size * position_size
            equity[i] += pnl

        # FTMO DD checks
        daily_dd   = (equity[i] - day_start_equity) / account
        total_dd   = (equity[i] - account)          / account
        if daily_dd < -max_daily_dd or total_dd < -max_total_dd:
            halted = True
            if position != 0:
                trades.append({'entry': entry_idx, 'exit': i,
                               'direction': position,
                               'pnl': equity[i] - equity[entry_idx],
                               'halted': True})
                position = 0
            continue

        # Signal change → close old, open new
        new_sig = sig[i]
        if new_sig != position:
            # Close
            if position != 0:
                gross_pnl = position * (close[i] - entry_price) / pip_size * position_size
                cost      = spread_cost / pip_size * position_size
                net_pnl   = gross_pnl - cost
                trades.append({'entry': entry_idx, 'exit': i,
                               'direction': position,
                               'pnl': net_pnl, 'halted': False})
                equity[i] += net_pnl - (position * (close[i] - close[i-1]) / pip_size * position_size)
            # Open
            if new_sig != 0:
                entry_price = close[i]
                entry_idx   = i
            position = new_sig

        peak_equity = max(peak_equity, equity[i])

    equity_series = pd.Series(equity, index=data.index)
    return equity_series, trades

print('✅ FTMO Backtester ready')

---
## 📊 32-METRIC PERFORMANCE ENGINE
Matches Rick's metric set from the BTC_3EMA notebook.

---

In [ ]:
def compute_metrics(equity_series, trades_list, params_dict=None):
    """
    Compute all 32 metrics per parameter combination.
    Returns a flat dict ready to append to grid_search_results.
    """
    r = {**(params_dict or {})}

    rets = equity_series.pct_change().dropna()
    n    = len(rets)
    ann  = 252  # trading days

    if n < 5 or equity_series.iloc[-1] == 0:
        return None

    total_return    = equity_series.iloc[-1] / equity_series.iloc[0] - 1
    ann_return      = (1 + total_return) ** (ann / max(n, 1)) - 1
    total_profit    = equity_series.iloc[-1] - equity_series.iloc[0]
    vol             = rets.std() * np.sqrt(ann)
    sharpe          = ann_return / vol if vol > 0 else 0

    # Sortino
    down_std = rets[rets < 0].std() * np.sqrt(ann)
    sortino  = ann_return / down_std if down_std > 0 else 0

    # Drawdown
    roll_max = equity_series.cummax()
    dd       = (equity_series - roll_max) / roll_max
    max_dd   = float(dd.min())

    # Calmar
    calmar = ann_return / abs(max_dd) if max_dd != 0 else 0

    # Ulcer Index
    ulcer = np.sqrt((dd ** 2).mean())

    # Omega Ratio
    pos_sum = rets[rets > 0].sum()
    neg_sum = abs(rets[rets < 0].sum())
    omega   = pos_sum / neg_sum if neg_sum > 0 else np.nan

    # Tail Ratio
    p95 = np.percentile(rets, 95)
    p05 = abs(np.percentile(rets, 5))
    tail_ratio = p95 / p05 if p05 > 0 else np.nan

    # Information Ratio (vs zero — standalone)
    ir = rets.mean() / rets.std() * np.sqrt(ann) if rets.std() > 0 else 0

    # Deflated Sharpe (Bailey & Lopez de Prado)
    skew = float(stats.skew(rets))
    kurt = float(stats.kurtosis(rets))
    sr_std = np.sqrt((1 + (skew * sharpe) - ((kurt - 1) / 4) * sharpe**2) / (n - 1))
    deflated_sharpe = stats.norm.cdf((sharpe - 0) / sr_std) if sr_std > 0 else 0

    # Trade metrics
    total_trades = len(trades_list)
    pnls = [t['pnl'] for t in trades_list]

    if total_trades == 0 or not pnls:
        return None

    wins   = [p for p in pnls if p > 0]
    losses = [p for p in pnls if p <= 0]
    win_rate    = len(wins) / total_trades if total_trades > 0 else 0
    profit_fac  = sum(wins) / abs(sum(losses)) if losses and sum(losses) != 0 else np.nan
    avg_win     = np.mean(wins)   if wins   else 0
    avg_loss    = np.mean(losses) if losses else 0
    payoff_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else np.nan
    expectancy  = win_rate * avg_win + (1 - win_rate) * avg_loss
    largest_win  = max(wins)   if wins   else 0
    largest_loss = min(losses) if losses else 0

    # SQN (System Quality Number — Van Tharp)
    pnl_series = pd.Series(pnls)
    sqn = np.sqrt(total_trades) * pnl_series.mean() / pnl_series.std() if pnl_series.std() > 0 else 0

    # Average trade duration
    durations = [t['exit'] - t['entry'] for t in trades_list]
    avg_dur   = np.mean(durations) if durations else 0

    # Streaks
    win_streak = los_streak = cur_w = cur_l = max_w = max_l = 0
    for p in pnls:
        if p > 0:
            cur_w += 1; cur_l = 0
        else:
            cur_l += 1; cur_w = 0
        max_w = max(max_w, cur_w)
        max_l = max(max_l, cur_l)

    # Recovery Factor
    recovery_factor = total_profit / abs(max_dd * ACCOUNT_SIZE) if max_dd != 0 else np.nan

    # Gain-to-Pain Ratio
    gpr = rets.sum() / abs(rets[rets < 0].sum()) if rets[rets < 0].sum() != 0 else np.nan

    # Serenity Index
    serenity = sharpe / ulcer if ulcer > 0 else np.nan

    # FTMO-specific
    ftmo_max_dd   = abs(max_dd) <= MAX_TOTAL_DD_PCT
    daily_rets    = equity_series.resample('D').last().pct_change().dropna()
    ftmo_daily_dd = (daily_rets.min() >= -MAX_DAILY_DD_PCT)
    ftmo_profit   = total_return >= PROFIT_TARGET_PCT
    ftmo_pass     = ftmo_max_dd and ftmo_daily_dd and ftmo_profit

    r.update({
        # Returns
        'total_return'      : round(total_return * 100, 3),
        'annualized_return' : round(ann_return   * 100, 3),
        'total_profit'      : round(total_profit, 2),
        # Risk-adjusted
        'sharpe_ratio'      : round(sharpe, 4),
        'sortino_ratio'     : round(sortino, 4),
        'calmar_ratio'      : round(calmar, 4),
        'omega_ratio'       : round(omega, 4) if not np.isnan(omega) else None,
        'information_ratio' : round(ir, 4),
        'tail_ratio'        : round(tail_ratio, 4) if not np.isnan(tail_ratio) else None,
        'deflated_sharpe'   : round(deflated_sharpe, 4),
        # Risk
        'max_drawdown'      : round(max_dd * 100, 3),
        'volatility'        : round(vol * 100, 3),
        'ulcer_index'       : round(ulcer * 100, 4),
        # Trade performance
        'win_rate'          : round(win_rate * 100, 2),
        'total_trades'      : total_trades,
        'avg_trade_duration': round(avg_dur, 1),
        'expectancy'        : round(expectancy, 4),
        'profit_factor'     : round(profit_fac, 4) if not np.isnan(profit_fac) else None,
        'sqn'               : round(sqn, 4),
        # Win/loss
        'payoff_ratio'      : round(payoff_ratio, 4) if not np.isnan(payoff_ratio) else None,
        'largest_win'       : round(largest_win, 2),
        'largest_loss'      : round(largest_loss, 2),
        'avg_win_amount'    : round(avg_win, 2),
        'avg_loss_amount'   : round(avg_loss, 2),
        'winning_streak'    : max_w,
        'losing_streak'     : max_l,
        # Additional
        'recovery_factor'   : round(recovery_factor, 4) if not np.isnan(recovery_factor) else None,
        'gain_to_pain_ratio': round(gpr, 4) if not np.isnan(gpr) else None,
        'serenity_index'    : round(serenity, 4) if not np.isnan(serenity) else None,
        # FTMO flags
        'ftmo_pass'         : ftmo_pass,
        'ftmo_profit_ok'    : ftmo_profit,
        'ftmo_total_dd_ok'  : ftmo_max_dd,
        'ftmo_daily_dd_ok'  : ftmo_daily_dd,
    })
    return r

print('✅ 32-metric Performance Engine ready')
print('Metrics to collect per combination:')
metrics_list = [
    'total_return','annualized_return','total_profit',
    'sharpe_ratio','sortino_ratio','calmar_ratio','omega_ratio','information_ratio','tail_ratio','deflated_sharpe',
    'max_drawdown','volatility','ulcer_index',
    'win_rate','total_trades','avg_trade_duration','expectancy','profit_factor','sqn',
    'payoff_ratio','largest_win','largest_loss','avg_win_amount','avg_loss_amount','winning_streak','losing_streak',
    'recovery_factor','gain_to_pain_ratio','serenity_index',
    'ftmo_pass','ftmo_profit_ok','ftmo_total_dd_ok','ftmo_daily_dd_ok'
]
for i, m in enumerate(metrics_list, 1):
    print(f'  {i:2d}. {m.replace("_", " ").title()}')

---
## 🔢 DEFINE PARAMETER RANGES FOR GRID SEARCH
---

In [ ]:
# ═══════════════════════════════════════════════════════
#  GRID SEARCH PARAMETER RANGES  — edit these freely
# ═══════════════════════════════════════════════════════

GRID_PARAMS = {

    'EMA Cross': {
        'fast'       : list(range(5,  40,  3)),   # [5, 8, 11, ..., 38]
        'slow'       : list(range(50, 200, 10)),   # [50, 60, ..., 190]
        'atr_period' : [14],                       # keep constant
        'atr_mult'   : [0.8, 1.0, 1.5],
    },

    'RSI Mean Revert': {
        'rsi_period' : list(range(10, 22, 2)),     # [10, 12, 14, 16, 18, 20]
        'oversold'   : [25, 30, 35],
        'overbought' : [65, 70, 75],
        'ema_trend'  : [100, 200],
    },

    'BB Breakout': {
        'bb_period'  : list(range(15, 35, 5)),     # [15, 20, 25, 30]
        'bb_std'     : [1.5, 2.0, 2.5],
        'rsi_period' : [14],
        'rsi_thresh' : [45, 50, 55],
    },

    'MACD Trend': {
        'fast'       : [8, 10, 12],
        'slow'       : [21, 26, 30],
        'signal'     : [7, 9, 12],
        'adx_period' : [14],
        'adx_thresh' : [20, 25, 30],
    },

    'Donchian': {
        'period'      : list(range(15, 55, 5)),    # [15, 20, 25, ..., 50]
        'exit_period' : list(range(5,  25, 5)),    # [5, 10, 15, 20]
    },
}

# ── Generate all valid combinations for active strategy ──
params     = GRID_PARAMS[ACTIVE_STRATEGY]
keys       = list(params.keys())
all_values = list(params.values())
all_combos = list(iproduct(*all_values))

# Apply fast < slow constraint where applicable
if 'fast' in keys and 'slow' in keys:
    fi = keys.index('fast')
    si = keys.index('slow')
    all_combos = [c for c in all_combos if c[fi] < c[si]]

print(f'Strategy: {ACTIVE_STRATEGY}')
print(f'Parameter ranges:')
for k, v in params.items():
    print(f'  {k}: {v}')
print(f'\n🔢 Total combinations to test: {len(all_combos):,}')
est_secs = len(all_combos) * 0.015
print(f'⏱️  Estimated runtime: {est_secs/60:.1f} – {est_secs*2/60:.1f} minutes')
print(f'\nReady to start the grid search!')

---
## ▶️ RUN GRID SEARCH — TRAINING SET
Tests every combination. Results stored in `grid_search_results`.

---

In [ ]:
grid_search_results = []
failed = 0
sig_fn = STRATEGY_SIGNALS[ACTIVE_STRATEGY]

print('=' * 70)
print(f'  GRID SEARCH — {ACTIVE_STRATEGY}  ({TICKER} Training Set)')
print(f'  Optimization Metric: Sharpe Ratio (risk-adjusted returns)')
print('=' * 70)
print(f'  Total combinations to test: {len(all_combos):,}')
print(f'  Min trades required: {MIN_TRADES}')
print()

BATCH_SIZE = 500
n_batches  = (len(all_combos) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx in range(n_batches):
    start = batch_idx * BATCH_SIZE
    end   = min(start + BATCH_SIZE, len(all_combos))
    batch = all_combos[start:end]

    print(f'Processing batch {batch_idx+1}/{n_batches}')
    print(f'  Combinations {start+1} to {end}...')

    batch_results = []
    for combo in batch:
        combo_params = dict(zip(keys, combo))
        try:
            signals  = sig_fn(train_data, **combo_params)
            equity, trades = run_ftmo_backtest(train_data, signals)
            if len(trades) < MIN_TRADES:
                continue
            result = compute_metrics(equity, trades, params_dict=combo_params)
            if result:
                batch_results.append(result)
        except Exception:
            failed += 1

    grid_search_results.extend(batch_results)
    print(f'  ✔ Batch complete: {len(batch_results)} successful')
    print(f'  Overall progress: {end}/{len(all_combos)} ({end/len(all_combos)*100:.1f}%)')
    print()

print('=' * 70)
print('  GRID SEARCH COMPLETED!')
print('=' * 70)
print(f'  Total combinations attempted : {len(all_combos):,}')
print(f'  Successfully completed       : {len(grid_search_results):,}')
print(f'  Failed / insufficient trades : {len(all_combos) - len(grid_search_results):,}')
print(f'  FTMO Pass rate               : {sum(1 for r in grid_search_results if r.get("ftmo_pass")) / max(len(grid_search_results),1)*100:.1f}%')
print()

results_df = pd.DataFrame(grid_search_results)
results_df = results_df.sort_values('sharpe_ratio', ascending=False).reset_index(drop=True)

print(f'✅ Results stored in grid_search_results ({len(grid_search_results):,} entries)')

---
## 🏆 TOP 5 IN-SAMPLE RESULTS
---

In [ ]:
print('=' * 70)
print('🏆 TOP 5 COMBINATIONS (by In-Sample Sharpe Ratio)')
print('=' * 70)

for rank, (_, row) in enumerate(results_df.head(5).iterrows(), 1):
    param_str = '  |  '.join(f'{k}={row[k]}' for k in keys)
    print(f'\n#{rank} - {ACTIVE_STRATEGY}({param_str})')
    print(f'   Sharpe Ratio:      {row["sharpe_ratio"]:.3f}')
    print(f'   Sortino Ratio:     {row["sortino_ratio"]:.3f}')
    print(f'   Total Return:      {row["total_return"]:.2f}%')
    print(f'   Annualized Return: {row["annualized_return"]:.2f}%')
    print(f'   Max Drawdown:      {row["max_drawdown"]:.2f}%')
    print(f'   Win Rate:          {row["win_rate"]:.1f}%')
    print(f'   Profit Factor:     {row["profit_factor"]}')
    print(f'   SQN:               {row["sqn"]:.3f}')
    print(f'   Total Trades:      {int(row["total_trades"])} ({int(row["total_trades"])/max((results_df["total_trades"].max()/10),1):.1f}/year est.)')
    print(f'   FTMO Pass:         {"✅ YES" if row["ftmo_pass"] else "❌ NO"}')

# Store best params globally
best_row    = results_df.iloc[0]
BEST_PARAMS = {k: best_row[k] for k in keys}
base_is_sharpe = best_row['sharpe_ratio']

print(f'\n✅ Best IS params saved: {BEST_PARAMS}')

---
## 📈 FULL RESULTS ANALYSIS
Distribution plots, Sharpe landscape, and FTMO pass heatmap.

---

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'{ACTIVE_STRATEGY} Grid Search Results — {TICKER} Training Set', fontsize=15, fontweight='bold')

# 1. Sharpe distribution
ax = axes[0, 0]
ax.hist(results_df['sharpe_ratio'].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(base_is_sharpe, color='gold', lw=2, label=f'Best: {base_is_sharpe:.3f}')
ax.axvline(1.0, color='lime', lw=1.5, ls='--', label='SR=1.0')
ax.set_xlabel('Sharpe Ratio'); ax.set_ylabel('Count')
ax.set_title('Sharpe Ratio Distribution'); ax.legend()

# 2. Return distribution
ax = axes[0, 1]
ax.hist(results_df['total_return'].dropna(), bins=40, color='mediumseagreen', edgecolor='white', alpha=0.85)
ax.axvline(0, color='white', lw=1)
ax.set_xlabel('Total Return (%)'); ax.set_ylabel('Count')
ax.set_title('Total Return Distribution')

# 3. Drawdown distribution
ax = axes[0, 2]
ax.hist(results_df['max_drawdown'].dropna(), bins=40, color='tomato', edgecolor='white', alpha=0.85)
ax.axvline(-MAX_TOTAL_DD_PCT * 100, color='yellow', lw=2, ls='--', label=f'FTMO limit: -{MAX_TOTAL_DD_PCT*100:.0f}%')
ax.set_xlabel('Max Drawdown (%)'); ax.set_ylabel('Count')
ax.set_title('Max Drawdown Distribution'); ax.legend()

# 4. Sharpe vs Return scatter
ax = axes[1, 0]
ftmo_mask = results_df['ftmo_pass'].fillna(False)
ax.scatter(results_df.loc[~ftmo_mask, 'total_return'], results_df.loc[~ftmo_mask, 'sharpe_ratio'],
           alpha=0.3, s=8, color='gray', label='FTMO Fail')
ax.scatter(results_df.loc[ftmo_mask, 'total_return'], results_df.loc[ftmo_mask, 'sharpe_ratio'],
           alpha=0.7, s=12, color='lime', label='FTMO Pass')
ax.set_xlabel('Total Return (%)'); ax.set_ylabel('Sharpe Ratio')
ax.set_title('Sharpe vs Return (green=FTMO pass)'); ax.legend(fontsize=8)

# 5. Win rate vs Profit factor
ax = axes[1, 1]
valid = results_df.dropna(subset=['win_rate','profit_factor'])
ax.scatter(valid['win_rate'], valid['profit_factor'], alpha=0.3, s=8, c=valid['sharpe_ratio'],
           cmap='RdYlGn', vmin=0, vmax=2)
ax.set_xlabel('Win Rate (%)'); ax.set_ylabel('Profit Factor')
ax.set_title('Win Rate vs Profit Factor\n(colour = Sharpe)')
ax.set_ylim(0, min(results_df['profit_factor'].quantile(0.99), 20))

# 6. Top 20 Sharpe bar
ax = axes[1, 2]
top20 = results_df.head(20)
labels = [str({k: v for k, v in zip(keys, [row[k] for k in keys])}) for _, row in top20.iterrows()]
short_labels = [f'#{i+1}' for i in range(len(top20))]
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(top20))]
ax.barh(short_labels[::-1], top20['sharpe_ratio'].values[::-1], color=colors[::-1], alpha=0.85)
ax.axvline(1.0, color='lime', lw=1.5, ls='--')
ax.set_xlabel('Sharpe Ratio'); ax.set_title('Top 20 Combinations')

for spine in ax.spines.values(): spine.set_color('#444444')

plt.tight_layout()
plt.show()

print(f'\n📊 Grid Search Summary:')
print(f'  Sharpe range  : {results_df["sharpe_ratio"].min():.3f} → {results_df["sharpe_ratio"].max():.3f}')
print(f'  FTMO pass rate: {ftmo_mask.mean()*100:.1f}%  ({ftmo_mask.sum()} / {len(results_df)} combos)')
print(f'  Median Sharpe : {results_df["sharpe_ratio"].median():.3f}')

---
## 🧪 OUT-OF-SAMPLE VALIDATION
Apply best IS parameters to untouched test data.

---

In [ ]:
print('=' * 70)
print('  OUT-OF-SAMPLE VALIDATION')
print('=' * 70)
print(f'  Best IS params: {BEST_PARAMS}')
print()

# Run OOS
oos_signals = sig_fn(test_data, **BEST_PARAMS)
oos_equity, oos_trades = run_ftmo_backtest(test_data, oos_signals)
oos_metrics = compute_metrics(oos_equity, oos_trades, params_dict=BEST_PARAMS)

# Run IS with best params (for side-by-side comparison)
is_signals = sig_fn(train_data, **BEST_PARAMS)
is_equity, is_trades = run_ftmo_backtest(train_data, is_signals)
is_metrics = compute_metrics(is_equity, is_trades, params_dict=BEST_PARAMS)

base_oos_sharpe = oos_metrics['sharpe_ratio'] if oos_metrics else float('nan')

print(f'{"Metric":<28} {"In-Sample":>14} {"Out-of-Sample":>16} {"Degradation":>14}')
print('-' * 74)

compare_metrics = [
    ('sharpe_ratio', 'Sharpe Ratio'),
    ('sortino_ratio', 'Sortino Ratio'),
    ('total_return', 'Total Return %'),
    ('annualized_return', 'Ann. Return %'),
    ('max_drawdown', 'Max Drawdown %'),
    ('win_rate', 'Win Rate %'),
    ('profit_factor', 'Profit Factor'),
    ('total_trades', 'Total Trades'),
    ('sqn', 'SQN'),
    ('ftmo_pass', 'FTMO Pass'),
]

for key, label in compare_metrics:
    is_val  = is_metrics.get(key, 'N/A') if is_metrics else 'N/A'
    oos_val = oos_metrics.get(key, 'N/A') if oos_metrics else 'N/A'
    if isinstance(is_val, float) and isinstance(oos_val, float) and is_val != 0:
        deg = (oos_val - is_val) / abs(is_val) * 100
        flag = '✅' if deg >= -20 else '⚠️' if deg >= -40 else '🔴'
        deg_str = f'{flag} {deg:+.1f}%'
    else:
        deg_str = ''
    is_str  = f'{is_val}' if isinstance(is_val, bool) else (f'{is_val:.3f}' if isinstance(is_val, float) else str(is_val))
    oos_str = f'{oos_val}' if isinstance(oos_val, bool) else (f'{oos_val:.3f}' if isinstance(oos_val, float) else str(oos_val))
    print(f'{label:<28} {is_str:>14} {oos_str:>16} {deg_str:>14}')

# Equity curve plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(f'IS vs OOS Equity — {ACTIVE_STRATEGY} Best Params {BEST_PARAMS}', fontsize=13, fontweight='bold')

for ax, eq, title, col in [
    (axes[0], is_equity,  f'In-Sample ({train_data.index[0].date()} → {train_data.index[-1].date()})',  'steelblue'),
    (axes[1], oos_equity, f'Out-of-Sample ({test_data.index[0].date()} → {test_data.index[-1].date()})', 'mediumseagreen'),
]:
    ax.plot(eq.index, eq, color=col, lw=1.8)
    ax.axhline(ACCOUNT_SIZE, color='white', lw=0.8, ls='--', alpha=0.5)
    ax.axhline(ACCOUNT_SIZE * (1 + PROFIT_TARGET_PCT), color='lime',   lw=1, ls=':', label=f'+{PROFIT_TARGET_PCT*100:.0f}% target')
    ax.axhline(ACCOUNT_SIZE * (1 - MAX_TOTAL_DD_PCT),  color='tomato', lw=1, ls=':', label=f'-{MAX_TOTAL_DD_PCT*100:.0f}% limit')
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Equity ($)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

---
## 🔬 PARAMETER SENSITIVITY ANALYSIS
Vary each parameter one at a time around the best. Bar charts show IS and OOS Sharpe Δ%.
🟩 Green = robust. 🟥 Red = fragile.

---

In [ ]:
print(f'Running sensitivity analysis around best params: {BEST_PARAMS}')
print(f'(Testing each parameter individually — all others held at best value)\n')

# Re-run best combination to get exact base values
is_base_signals  = sig_fn(train_data, **BEST_PARAMS)
is_base_eq, is_base_trades = run_ftmo_backtest(train_data, is_base_signals)
is_base_m = compute_metrics(is_base_eq, is_base_trades)
base_is_sharpe  = is_base_m['sharpe_ratio'] if is_base_m else 0

oos_base_signals = sig_fn(test_data, **BEST_PARAMS)
oos_base_eq, oos_base_trades = run_ftmo_backtest(test_data, oos_base_signals)
oos_base_m = compute_metrics(oos_base_eq, oos_base_trades)
base_oos_sharpe = oos_base_m['sharpe_ratio'] if oos_base_m else float('nan')

# For each parameter, vary it and hold others fixed
sensitivity_results = {k: [] for k in keys}

for param_key in keys:
    print(f'  Varying {param_key}...')
    for val in params[param_key]:
        test_params = {**BEST_PARAMS, param_key: val}
        # Enforce fast < slow if relevant
        if 'fast' in test_params and 'slow' in test_params:
            if test_params['fast'] >= test_params['slow']:
                continue
        try:
            is_sig  = sig_fn(train_data, **test_params)
            oos_sig = sig_fn(test_data,  **test_params)
            is_eq,  is_t  = run_ftmo_backtest(train_data, is_sig)
            oos_eq, oos_t = run_ftmo_backtest(test_data,  oos_sig)
            is_m  = compute_metrics(is_eq,  is_t)
            oos_m = compute_metrics(oos_eq, oos_t)
            sensitivity_results[param_key].append({
                'value'     : val,
                'is_sharpe' : is_m['sharpe_ratio']  if is_m  else np.nan,
                'oos_sharpe': oos_m['sharpe_ratio'] if oos_m else np.nan,
            })
        except Exception:
            pass

print('\n✅ Sensitivity data collected. Plotting...')

# ── PLOT: 2 rows × N_params cols  (row 0 = IS, row 1 = OOS) ──
n_params = len(keys)
fig, axes = plt.subplots(2, n_params, figsize=(6 * n_params, 10))
if n_params == 1:
    axes = axes.reshape(2, 1)
fig.suptitle(f'Parameter Sensitivity — Base: {BEST_PARAMS}', fontsize=14, fontweight='bold')

def sharpe_delta_colors(deltas):
    cols = []
    for d in deltas:
        if   d < -40: cols.append('red')
        elif d <   0: cols.append('orange')
        elif d <  10: cols.append('lightgreen')
        else:          cols.append('darkgreen')
    return cols

for ci, param_key in enumerate(keys):
    data_rows = sensitivity_results[param_key]
    if not data_rows:
        continue
    df_s = pd.DataFrame(data_rows)

    best_val = BEST_PARAMS[param_key]

    for row_idx, (sharpe_col, split_label, base_sharpe) in enumerate([
        ('is_sharpe',  'In-Sample',     base_is_sharpe),
        ('oos_sharpe', 'Out-of-Sample', base_oos_sharpe),
    ]):
        ax = axes[row_idx, ci]
        deltas = ((df_s[sharpe_col] - base_sharpe) / abs(base_sharpe) * 100).fillna(0).tolist()
        colors = sharpe_delta_colors(deltas)

        bars = ax.bar(df_s['value'].astype(str), deltas, color=colors, edgecolor='black', linewidth=0.5)
        ax.axhline(0, color='black', linewidth=1.5)
        # Mark best value
        x_labels = df_s['value'].astype(str).tolist()
        if str(best_val) in x_labels:
            bi = x_labels.index(str(best_val))
            ax.get_xticklabels()
            ax.axvline(bi, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Best={best_val}')

        ax.set_xlabel(param_key.replace('_', ' ').title(), fontsize=10, fontweight='bold')
        ax.set_ylabel('Sharpe Δ%', fontsize=10, fontweight='bold')
        ax.set_title(f'{param_key} ({split_label})', fontsize=11, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        ax.legend(fontsize=9)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

# ── Sensitivity Summary Table ──
print('\n📋 SENSITIVITY SUMMARY')
print('=' * 80)
summary_rows = []
for param_key in keys:
    data_rows = sensitivity_results[param_key]
    if not data_rows:
        continue
    df_s     = pd.DataFrame(data_rows)
    is_range = f'{df_s["is_sharpe"].min():.3f} – {df_s["is_sharpe"].max():.3f}'
    is_delta = df_s['is_sharpe'].min() - base_is_sharpe
    oos_range = f'{df_s["oos_sharpe"].min():.3f} – {df_s["oos_sharpe"].max():.3f}'
    oos_delta = df_s['oos_sharpe'].min() - base_oos_sharpe if not np.isnan(base_oos_sharpe) else np.nan
    sensitivity_flag = '⚠️  HIGH' if abs(is_delta) > 0.4 * base_is_sharpe else '✅ LOW'
    summary_rows.append({
        'Parameter'    : param_key,
        'IS Range'     : is_range,
        'IS Min Δ'     : f'{is_delta:+.3f}',
        'OOS Range'    : oos_range,
        'OOS Min Δ'    : f'{oos_delta:+.3f}' if not np.isnan(oos_delta) else 'N/A',
        'Sensitivity'  : sensitivity_flag,
    })

print(pd.DataFrame(summary_rows).to_string(index=False))
print('\n✅ Analysis Complete! Green = Robust  |  Red = Fragile')

---
## 💾 EXPORT RESULTS
---

In [ ]:
ticker_clean = TICKER.replace('=X','').replace('-','_')
strat_clean  = ACTIVE_STRATEGY.replace(' ', '_')
prefix       = f'{ticker_clean}_{strat_clean}'

results_df.to_csv(f'{prefix}_grid_results.csv', index=False)
is_equity.to_csv(f'{prefix}_is_equity.csv', header=['equity'])
oos_equity.to_csv(f'{prefix}_oos_equity.csv', header=['equity'])

top5 = results_df.head(5)
top5.to_csv(f'{prefix}_top5.csv', index=False)

print('✅ Exported:')
print(f'  {prefix}_grid_results.csv  ({len(results_df):,} combinations)')
print(f'  {prefix}_is_equity.csv')
print(f'  {prefix}_oos_equity.csv')
print(f'  {prefix}_top5.csv')